<a href="https://colab.research.google.com/github/maierav/claupenscope/blob/main/notebooks/openscope_pp_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenScope Community Predictive Processing — Analysis Demo

This notebook demonstrates core analyses from the **openscope-pp** package:

| Section | Dataset | What we show |
|---|---|---|
| **1 — RF Mapping** | Mesoscope (dandiset 001768, sub-839909) | Population PSTH + per-ROI receptive field maps for VISp plane 0 |
| **2 — Oddball Mismatch** | Ecephys (dandiset 001637, sub-820459) | SUA responses to standard vs deviant stimuli, split by deviant type |
| **3 — Orientation Tuning** | Mesoscope (dandiset 001768, sub-839909) | Polar tuning curves + OSI/DSI for VISp vs VISl (single session) |
| **4 — Population Summary** | All 3 techniques (precomputed) | Cross-session OSI/DSI comparison: mesoscope, ecephys, SLAP2 |

All data stream directly from DANDI — nothing is downloaded locally.  
Sections 3–4 load precomputed per-session statistics from the repository.

> **Runtime estimate:** ~8 min on a standard Colab CPU instance.

In [ ]:
# Install dependencies (run once per session)
!pip install -q git+https://github.com/maierav/claupenscope.git \
    dandi remfile h5py pynwb scipy matplotlib numpy pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import h5py, remfile, time

from openscope_pp.loaders.streaming import open_nwb
from openscope_pp.loaders.trials import load_trials

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Imports OK")

---
## Section 1 — Mesoscope Receptive Field Mapping

**Dataset:** dandiset 001768, sub-839909 (3.6 GB NWB, streamed via remfile)  
**Stimuli:** 9 × 9 spatial grid (±40°), 15 trials/position, 1215 trials total  
**Analysis plane:** VISp_0 — 350 soma-only ROIs at ~10.7 Hz (GCaMP)

Steps:
1. Stream the NWB file and load RF trial times + grid positions
2. Bulk-read the dFF span covering all trials; interpolate per-ROI snippets
3. Z-score, average over trials per grid position → RF map per ROI
4. Plot population PSTH (alignment check) and top individual RF maps

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
MESO_ASSET_ID = "00451971-f839-4cb0-a11b-1d2e12c8ad6b"   # sub-839909
PLANE         = "VISp_0"
WINDOW        = (-0.5, 1.5)     # s around stimulus onset
RESP_WIN      = (0.1, 0.8)      # response scoring window (slow GCaMP dynamics)
RF_SMOOTH     = 0.8             # gaussian sigma in grid units
N_TOP         = 12              # individual ROI maps to show

# ── Open streaming NWB ────────────────────────────────────────────────
print("Connecting to DANDI…")
t0 = time.time()
handle = open_nwb(MESO_ASSET_ID)
h5 = handle.h5
print(f"  Opened in {time.time()-t0:.1f}s  (technique: {handle.technique})")

# ── Load RF trial table ───────────────────────────────────────────────
def decode_col(arr):
    return np.array([float(v.decode() if isinstance(v, bytes) else v) for v in arr])

rf_grp  = h5["intervals"]["RF mapping_presentations"]
starts  = rf_grp["start_time"][:]
xs_raw  = decode_col(rf_grp["X"][:])
ys_raw  = decode_col(rf_grp["Y"][:])
xs_grid = np.array(sorted(np.unique(xs_raw)))
ys_grid = np.array(sorted(np.unique(ys_raw)))

print(f"RF mapping: {len(starts)} trials, {len(xs_grid)}×{len(ys_grid)} grid, "
      f"{len(starts)/(len(xs_grid)*len(ys_grid)):.0f} trials/position")
print(f"  X: {xs_grid}°")
print(f"  Y: {ys_grid}°")

# ── Load dFF + soma mask for chosen plane ─────────────────────────────
base     = f"processing/{PLANE}"
dff_path = f"{base}/dff_timeseries/dff_timeseries"
ts       = h5[f"{dff_path}/timestamps"][:]
data     = h5[f"{dff_path}/data"]            # lazy HDF5 dataset
is_soma  = h5[f"{base}/image_segmentation/roi_table/is_soma"][:].astype(bool)
n_soma   = is_soma.sum()
print(f"\n{PLANE}: {data.shape[1]} ROIs → {n_soma} soma  |  rate: "
      f"{1/float(np.median(np.diff(ts[len(ts)//2:len(ts)//2+200]))):.1f} Hz")

In [ ]:
# ── Extract snippets via np.interp (robust to gaps / clock drift) ─────
valid   = (starts + WINDOW[0] >= ts[0]) & (starts + WINDOW[1] <= ts[-1])
onsets  = starts[valid]
xs_v    = xs_raw[valid]
ys_v    = ys_raw[valid]
print(f"{valid.sum()}/{len(starts)} RF trials within recording window")

dt       = float(np.median(np.diff(ts[len(ts)//2 : len(ts)//2 + 200])))
n_samp   = int(round((WINDOW[1] - WINDOW[0]) / dt))
t_rel    = np.linspace(WINDOW[0], WINDOW[0] + (n_samp - 1) * dt, n_samp)
tc       = t_rel + dt / 2

# Bulk HDF5 read covering the full RF block
t0_span = onsets.min() + WINDOW[0] - 2.0
t1_span = onsets.max() + WINDOW[1] + 2.0
i0 = max(0, int(np.searchsorted(ts, t0_span)))
i1 = min(data.shape[0], int(np.searchsorted(ts, t1_span)) + 1)
print(f"Reading HDF5 span [{i0}:{i1}] = {i1-i0} samples × {data.shape[1]} ROIs …")
trace      = data[i0:i1, :].astype(np.float32)
ts_span    = ts[i0:i1]
trace_soma = trace[:, is_soma]   # keep soma only

# Interpolate: (n_trials, n_soma, n_samples)
t_query = onsets[:, None] + t_rel[None, :]   # (n_trials, n_samp)
t_flat  = t_query.ravel()
snip    = np.full((len(onsets), n_soma, n_samp), np.nan, dtype=np.float32)
for roi in range(n_soma):
    y   = trace_soma[:, roi]
    fin = np.isfinite(y)
    if fin.sum() < 10: continue
    vals = np.interp(t_flat, ts_span[fin], y[fin], left=np.nan, right=np.nan)
    snip[:, roi, :] = vals.reshape(len(onsets), n_samp)
print(f"Snippets shape: {snip.shape}")

# Z-score per ROI using pre-stimulus baseline
bl_mask = (tc >= -0.5) & (tc < 0.0)
mu  = np.nanmean(snip[:, :, bl_mask], axis=(0, 2), keepdims=True)
sig = np.nanstd( snip[:, :, bl_mask], axis=(0, 2), keepdims=True)
sig = np.where(sig > 1e-6, sig, 1.0)
z   = (snip - mu) / sig

# Build RF maps: (n_soma, n_y, n_x)
resp_mask = (tc >= RESP_WIN[0]) & (tc < RESP_WIN[1])
rf_maps   = np.zeros((n_soma, len(ys_grid), len(xs_grid)), dtype=np.float32)
for xi, x in enumerate(xs_grid):
    for yi, y in enumerate(ys_grid):
        m = (xs_v == x) & (ys_v == y)
        if m.sum() == 0: continue
        resp = np.nanmean(z[m][:, :, resp_mask], axis=(0, 2))
        base = np.nanmean(z[m][:, :, bl_mask  ], axis=(0, 2))
        rf_maps[:, yi, xi] = resp - base

peak = np.array([gaussian_filter(rf_maps[u], RF_SMOOTH).max() for u in range(n_soma)])
print(f"Peak z-score range: [{peak.min():.3f}, {peak.max():.3f}]")

In [ ]:
# ── Figure 1a: Population PSTH ─────────────────────────────────────────
pop = np.nanmean(z, axis=1)           # (trial, time)
m   = np.nanmean(pop, axis=0)
s   = np.nanstd( pop, axis=0) / np.sqrt(np.isfinite(pop).sum(axis=0).clip(1))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.axvspan(0, 0.25, color="gray", alpha=0.12, label="stim on (250 ms)")
ax.axhline(0, color="gray", lw=0.5, ls="--")
ax.axvline(0, color="gray", lw=0.8, ls=":", label="stimulus onset")
ax.fill_between(tc, m - s, m + s, color="#4878CF", alpha=0.3)
ax.plot(tc, m, color="#4878CF", lw=2)

bl_mu = np.nanmean(m[tc < 0]); bl_sd = np.nanstd(m[tc < 0])
first = np.where((tc > 0) & (m > bl_mu + 2*bl_sd))[0]
if len(first):
    lat = tc[first[0]]
    ax.axvline(lat, color="red", lw=1.2, ls="--")
    ax.text(lat + 0.05, m.max(), f"≈{lat*1000:.0f} ms", color="red", fontsize=9)

ax.set(xlabel="Time from onset (s)", ylabel="Mean z-score",
       title=f"Mesoscope {PLANE} — population PSTH  ({n_soma} soma, {valid.sum()} RF trials)",
       xlim=WINDOW)
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

# ── Figure 1b: Top individual RF maps ─────────────────────────────────
order   = np.argsort(peak)[::-1][:N_TOP]
all_sm  = np.array([gaussian_filter(rf_maps[u], RF_SMOOTH) for u in range(n_soma)])
vmax    = np.percentile(np.abs(all_sm[order]), 98)
ext     = [xs_grid[0]-5, xs_grid[-1]+5, ys_grid[0]-5, ys_grid[-1]+5]

n_cols = 4
n_rows = int(np.ceil(N_TOP / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*2.8, n_rows*2.8), squeeze=False)

for plot_i in range(n_rows * n_cols):
    row, col = divmod(plot_i, n_cols)
    ax = axes[row, col]
    if plot_i >= N_TOP:
        ax.set_visible(False); continue
    uid = order[plot_i]
    sm  = gaussian_filter(rf_maps[uid], RF_SMOOTH)
    im  = ax.imshow(sm, origin="lower", cmap="RdBu_r",
                    extent=ext, vmin=-vmax, vmax=vmax, aspect="equal")
    pk  = np.unravel_index(np.argmax(sm), sm.shape)
    ax.plot(xs_grid[pk[1]], ys_grid[pk[0]], "k+", ms=8, mew=1.5)
    ax.set(xticks=xs_grid[::3], yticks=ys_grid[::3])
    ax.tick_params(labelsize=6)
    ax.set_title(f"ROI {uid}  z={peak[uid]:.2f}", fontsize=8, pad=2)
    if col == 0: ax.set_ylabel("Y (°)", fontsize=7)
    if row == n_rows-1: ax.set_xlabel("X (°)", fontsize=7)

cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label="z-score vs baseline")
fig.suptitle(f"Mesoscope {PLANE} — top {N_TOP} soma RF maps  (sub-839909)",
             fontsize=10, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.92, 0.95])
plt.show()

---
## Section 2 — Ecephys Oddball Mismatch Responses

**Dataset:** dandiset 001256 (ecephys, Neuropixels)  
**Pipeline:**
1. Filter to **single units (SUA)** with a clear visual response (RF peak ≥ 5 spk/s above baseline)
2. Align spikes to each trial onset in a ±0.5–1.0 s window
3. Z-score against standard-trial baseline
4. Plot one PSTH line per trial type: standard, halt, omission, orientation deviants

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
EPHYS_ASSET_ID = "cd175e65-8faa-4216-86af-c1fd30e571a1"
RF_WINDOW    = (-0.1,  0.5)   # for visual unit selection
RF_BIN       = 0.01
RESP_WIN     = (0.03, 0.25)   # response window for RF peak
RESP_THRESH  = 5.0            # spk/s — visual unit threshold
RF_SMOOTH_E  = 0.8
ODD_WINDOW   = (-0.5,  1.0)
ODD_BIN      = 0.025
N_STD_MAX    = 300            # cap standard trials to balance sampling

COLORS = {
    "standard":         "#4878CF",
    "halt":             "#D65F5F",
    "omission":         "#E07B39",
    "orientation_45":   "#6BAF6B",
    "orientation_90":   "#9B59B6",
    "sequence_omission":"#C0392B",
}

# ── Load ──────────────────────────────────────────────────────────────
print("Opening ecephys NWB…")
t0 = time.time()
ehandle = open_nwb(EPHYS_ASSET_ID)
trials  = load_trials(ehandle)
print(f"  Opened in {time.time()-t0:.1f}s  |  {len(trials)} total stimulus presentations")

eh5        = ehandle.h5
units_grp  = eh5["units"]
n_total    = int(units_grp["id"].shape[0])
dl         = units_grp["decoder_label"][:]
dl         = np.array([v.decode() if isinstance(v, bytes) else str(v) for v in dl])
sua_idx    = np.where(dl == "sua")[0]
all_spikes = units_grp["spike_times"][:]
spk_index  = units_grp["spike_times_index"][:]

print(f"  {n_total} units total, {len(sua_idx)} SUA")

# ── Helper: bin spikes → (n_trials, n_units, n_bins) firing rates ─────
def bin_spikes(uid_arr, spikes, index, onsets, window, bin_size):
    pre, post = window
    edges   = np.arange(pre, post + bin_size, bin_size)
    centers = 0.5 * (edges[:-1] + edges[1:])
    out = np.zeros((len(onsets), len(uid_arr), len(centers)), dtype=np.float32)
    for j, uid in enumerate(uid_arr):
        i0  = int(index[uid-1]) if uid > 0 else 0
        i1  = int(index[uid])
        spk = spikes[i0:i1]
        for i, t0 in enumerate(onsets):
            rel = spk - t0
            in_w = rel[(rel >= pre) & (rel < post)]
            if len(in_w):
                cnt, _ = np.histogram(in_w, bins=edges)
                out[i, j, :] = cnt / bin_size
    return out, centers

In [ ]:
# ── Step 1: find visual units via RF mapping ──────────────────────────
print("Identifying visual units via RF mapping block…")
rf_t = trials[trials["block_kind"] == "rf_mapping"].reset_index(drop=True)

t1 = time.time()
rf_arr, rf_centers = bin_spikes(sua_idx, all_spikes, spk_index,
                                 rf_t["start_time"].values, RF_WINDOW, RF_BIN)
print(f"  Binned in {time.time()-t1:.1f}s")

xs_e = np.array(sorted(rf_t["x"].unique()))
ys_e = np.array(sorted(rf_t["y"].unique()))
bl_m = rf_centers < 0
rsp  = (rf_centers >= RESP_WIN[0]) & (rf_centers < RESP_WIN[1])
delta = rf_arr[:, :, rsp].mean(axis=2) - rf_arr[:, :, bl_m].mean(axis=2)

rf_maps_e = np.zeros((len(sua_idx), len(ys_e), len(xs_e)), dtype=np.float32)
for xi, x in enumerate(xs_e):
    for yi, y in enumerate(ys_e):
        m = (rf_t["x"].values == x) & (rf_t["y"].values == y)
        rf_maps_e[:, yi, xi] = delta[m, :].mean(axis=0)

peak_e   = np.array([gaussian_filter(rf_maps_e[u], RF_SMOOTH_E).max() for u in range(len(sua_idx))])
vis_mask = peak_e >= RESP_THRESH
vis_idx  = sua_idx[vis_mask]
n_vis    = len(vis_idx)

print(f"\n  Unit summary:")
print(f"    All units : {n_total}")
print(f"    SUA       : {len(sua_idx)}  ({100*len(sua_idx)/n_total:.0f}% of all)")
print(f"    Visual SUA: {n_vis}  ({100*n_vis/len(sua_idx):.0f}% of SUA)")

# ── Step 2: select oddball block, split by trial type ─────────────────
odd_t      = trials[trials["block_kind"] == "paradigm_oddball"]
first_blk  = odd_t["block"].unique()[0]
ob         = odd_t[odd_t["block"] == first_blk]
dev_types  = sorted(ob[ob["is_deviant"]]["trial_type"].unique().tolist())
trial_types = ["standard"] + dev_types
print(f"\n  Trial types in '{first_blk}': {trial_types}")

type_trials = {}
for tt in trial_types:
    sub = ob[ob["trial_type"] == tt]
    if tt == "standard" and len(sub) > N_STD_MAX:
        sub = sub.sample(N_STD_MAX, random_state=42)
    type_trials[tt] = sub.reset_index(drop=True)
    print(f"    {tt:20s}: {len(type_trials[tt])} trials")

# ── Step 3: bin oddball spikes per type ───────────────────────────────
print("\nBinning oddball spikes…")
type_arrays = {}
t1 = time.time()
for tt, sub in type_trials.items():
    arr, centers = bin_spikes(vis_idx, all_spikes, spk_index,
                               sub["start_time"].values, ODD_WINDOW, ODD_BIN)
    type_arrays[tt] = arr
print(f"  Done in {time.time()-t1:.1f}s")

# Z-score: baseline from standard-trial distribution
std_arr = type_arrays["standard"]
bl_t    = (centers >= -0.5) & (centers < 0.0)
mu_z    = std_arr[:, :, bl_t].mean(axis=(0, 2), keepdims=True)
sig_z   = std_arr[:, :, bl_t].std(axis=(0, 2), keepdims=True)
sig_z   = np.where(sig_z > 0.1, sig_z, 1.0)
type_z  = {tt: (arr - mu_z) / sig_z for tt, arr in type_arrays.items()}
print("Z-scored.")

In [ ]:
# ── Figure 2: Oddball PSTH split by deviant type ─────────────────────
def mean_sem(arr):
    per_trial = arr.mean(axis=1)   # (trial, time)
    return per_trial.mean(axis=0), per_trial.std(axis=0) / np.sqrt(per_trial.shape[0])

fig, ax = plt.subplots(figsize=(9, 5))

ax.axvspan(0, 0.25, color="gray", alpha=0.10, label="stim (250 ms)")
ax.axhline(0,    color="gray", lw=0.6, ls="--")
ax.axvline(0,    color="gray", lw=0.8, ls=":")
ax.axvline(0.25, color="gray", lw=0.6, ls=":")

# Standard (background reference)
m, s = mean_sem(type_z["standard"])
ax.fill_between(centers, m-s, m+s, color=COLORS["standard"], alpha=0.20)
ax.plot(centers, m, color=COLORS["standard"], lw=2.5,
        label=f"standard  (n={len(type_trials['standard'])})", zorder=3)

# Deviant types
for tt in dev_types:
    color = COLORS.get(tt, "#888888")
    m, s  = mean_sem(type_z[tt])
    n     = len(type_trials[tt])
    ax.fill_between(centers, m-s, m+s, color=color, alpha=0.18)
    ax.plot(centers, m, color=color, lw=2, label=f"{tt}  (n={n})", zorder=4)

ax.set_xlabel("Time from stimulus onset (s)", fontsize=11)
ax.set_ylabel("Population response (z-score)", fontsize=11)
ax.set_title(
    f"Ecephys oddball PSTH — {first_blk} — deviant types\n"
    f"Visual SUA: {n_vis}/{len(sua_idx)} ({100*n_vis/len(sua_idx):.0f}% of SUA, "
    f"{100*n_vis/n_total:.0f}% of all units)",
    fontsize=10
)
ax.set_xlim(ODD_WINDOW)
ax.legend(fontsize=9, framealpha=0.8, loc="upper right")
plt.tight_layout()
plt.show()

ehandle.close()
print("Done.")

---
## Section 3 — Mesoscope Orientation Tuning (single session)

**Dataset:** dandiset 001768, sub-839909  
**Stimuli:** Control block 2 — 14 directions × 80 trials each, orientations in radians  
**Areas:** VISp (planes 0–3) and VISl (planes 4–7)

Steps:
1. Load the orientation block trial table from the same streamed NWB
2. For each area, select the plane with the highest median OSI (best plane)
3. Compute per-ROI tuning curves → OSI and DSI via vector-sum method
4. Plot polar tuning curves for top 12 ROIs per area, and OSI/DSI histograms

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
ORI_ASSET_ID = "00451971-f839-4cb0-a11b-1d2e12c8ad6b"   # sub-839909 (same as Section 1)
AREAS = {
    "VISp": ["VISp_0", "VISp_1", "VISp_2", "VISp_3"],
    "VISl": ["VISl_4", "VISl_5", "VISl_6", "VISl_7"],
}
AREA_COLORS = {"VISp": "#4878CF", "VISl": "#D65F5F"}
ORI_WINDOW  = (-0.3, 1.2)   # wide to capture slow GCaMP dynamics
RESP_WIN    = (0.1,  0.8)
BL_WIN      = (-0.25, 0.0)
N_POLAR     = 12            # top ROIs per area to show

# ── Open NWB (reuse connection if already open, else reconnect) ────────
print("Connecting to dandiset 001768, sub-839909…")
t0 = time.time()
ori_handle = open_nwb(ORI_ASSET_ID)
ori_h5     = ori_handle.h5
print(f"  Opened in {time.time()-t0:.1f}s")

# ── Load orientation block ─────────────────────────────────────────────
g        = ori_h5["intervals"]["Control block 2_presentations"]
starts   = g["start_time"][:]
oris_rad = np.array([float(v.decode() if isinstance(v, bytes) else v)
                     for v in g["Orientation"][:]])
dirs_rad = np.array(sorted(np.unique(oris_rad)))
dirs_deg = np.degrees(dirs_rad)
print(f"\nOrientation block: {len(starts)} trials, {len(dirs_rad)} directions")
print(f"  Directions (°): {dirs_deg.round(1)}")
print(f"  Trials/dir:     {len(starts)/len(dirs_rad):.0f}")

# ── Helper: process one plane → tuning curves + OSI/DSI ──────────────
def process_plane(plane):
    base     = f"processing/{plane}"
    dff_path = f"{base}/dff_timeseries/dff_timeseries"
    ts       = ori_h5[f"{dff_path}/timestamps"][:]
    data     = ori_h5[f"{dff_path}/data"]
    is_soma  = ori_h5[f"{base}/image_segmentation/roi_table/is_soma"][:].astype(bool)

    pre, post = ORI_WINDOW
    dt        = float(np.median(np.diff(ts[len(ts)//2 : len(ts)//2 + 200])))
    n_samp    = int(round((post - pre) / dt))
    t_rel     = np.linspace(pre, pre + (n_samp - 1) * dt, n_samp)
    tc        = t_rel + dt / 2

    valid  = (starts + pre >= ts[0]) & (starts + post <= ts[-1])
    ons_v  = starts[valid];  ori_v = oris_rad[valid]

    i0 = max(0, int(np.searchsorted(ts, ons_v.min() + pre - 2.0)))
    i1 = min(data.shape[0], int(np.searchsorted(ts, ons_v.max() + post + 2.0)) + 1)
    trace      = data[i0:i1, :].astype(np.float32)
    ts_span    = ts[i0:i1]
    trace_soma = trace[:, is_soma]
    n_soma     = trace_soma.shape[1]

    t_flat = (ons_v[:, None] + t_rel[None, :]).ravel()
    snip   = np.full((len(ons_v), n_soma, n_samp), np.nan, dtype=np.float32)
    for roi in range(n_soma):
        y = trace_soma[:, roi]; fin = np.isfinite(y)
        if fin.sum() < 10: continue
        vals = np.interp(t_flat, ts_span[fin], y[fin], left=np.nan, right=np.nan)
        snip[:, roi, :] = vals.reshape(len(ons_v), n_samp)

    bl_m   = (tc >= BL_WIN[0])   & (tc < BL_WIN[1])
    rsp_m  = (tc >= RESP_WIN[0]) & (tc < RESP_WIN[1])
    mu     = np.nanmean(snip[:, :, bl_m],  axis=(0, 2), keepdims=True)
    sig    = np.nanstd( snip[:, :, bl_m],  axis=(0, 2), keepdims=True)
    z      = (snip - mu) / np.where(sig > 1e-6, sig, 1.0)

    curves = np.full((n_soma, len(dirs_rad)), np.nan, dtype=np.float32)
    for di, d in enumerate(dirs_rad):
        m = ori_v == d
        if m.sum() == 0: continue
        curves[:, di] = np.nanmean(z[m][:, :, rsp_m], axis=(0, 2)) - \
                        np.nanmean(z[m][:, :, bl_m  ], axis=(0, 2))

    r_sum = np.nansum(np.abs(curves), axis=1).clip(1e-9)
    r_ori = np.nansum(curves * np.exp(2j * dirs_rad)[None, :], axis=1)
    r_dir = np.nansum(curves * np.exp(1j * dirs_rad)[None, :], axis=1)
    OSI   = (np.abs(r_ori) / r_sum).real
    DSI   = (np.abs(r_dir) / r_sum).real
    pref_ori = (np.angle(r_ori) / 2) % np.pi
    return curves, OSI, DSI, pref_ori, n_soma

# ── Process all planes, pick best per area ─────────────────────────────
print("\nProcessing planes…")
results = {}
for area_name, planes in AREAS.items():
    for plane in planes:
        print(f"  {plane}…", end=" ", flush=True)
        curves, OSI, DSI, pref_ori, n_soma = process_plane(plane)
        results[plane] = dict(curves=curves, OSI=OSI, DSI=DSI,
                              pref_ori=pref_ori, n_soma=n_soma)
        print(f"OSI median={np.median(OSI):.3f}  n_soma={n_soma}")

for area_name, planes in AREAS.items():
    best = max(planes, key=lambda p: np.median(results[p]["OSI"]))
    print(f"  Best {area_name} plane: {best}  (median OSI={np.median(results[best]['OSI']):.3f})")

In [ ]:
# ── Figure 3a: Polar tuning curves — top N_POLAR ROIs per area ────────
theta = np.append(dirs_rad, dirs_rad[0])
cmap  = plt.cm.hsv

fig, all_axes = plt.subplots(
    len(AREAS) * int(np.ceil(N_POLAR / 6)), 6,
    figsize=(13, len(AREAS) * int(np.ceil(N_POLAR / 6)) * 2.2),
    subplot_kw={"projection": "polar"}, squeeze=False,
)
row_offset = 0
for area_name, planes in AREAS.items():
    best  = max(planes, key=lambda p: np.median(results[p]["OSI"]))
    r     = results[best]
    n_show = min(N_POLAR, r["n_soma"])
    order  = np.argsort(r["OSI"])[::-1][:n_show]
    n_rows_area = int(np.ceil(N_POLAR / 6))

    for plot_i in range(n_rows_area * 6):
        row = row_offset + plot_i // 6
        col = plot_i % 6
        ax  = all_axes[row, col]
        if plot_i >= n_show:
            ax.set_visible(False); continue
        uid   = order[plot_i]
        c     = np.clip(np.append(r["curves"][uid], r["curves"][uid][0]), 0, None)
        color = cmap(r["pref_ori"][uid] / np.pi)
        ax.plot(theta, c, lw=1.5, color=color)
        ax.fill(theta, c, alpha=0.25, color=color)
        ax.set(xticks=[], yticks=[])
        ax.set_title(f"[{area_name}] ROI {uid}\nOSI={r['OSI'][uid]:.2f}",
                     fontsize=6, pad=2, color=AREA_COLORS[area_name])
    row_offset += n_rows_area

fig.suptitle(
    f"Mesoscope orientation tuning — top {N_POLAR} soma ROIs per area  (sub-839909)\n"
    f"Best VISp plane vs best VISl plane · color = preferred orientation",
    fontsize=10, fontweight="bold",
)
plt.tight_layout()
plt.show()

# ── Figure 3b: OSI / DSI histograms, VISp vs VISl (pooled across planes) ──
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
bins = np.linspace(0, 1, 16)

for area_name, planes in AREAS.items():
    color = AREA_COLORS[area_name]
    OSI_pool = np.concatenate([results[p]["OSI"] for p in planes])
    DSI_pool = np.concatenate([results[p]["DSI"] for p in planes])
    n_pool   = len(OSI_pool)
    axes[0].hist(OSI_pool, bins=bins, color=color, alpha=0.6,
                 label=f"{area_name}  n={n_pool}  median={np.median(OSI_pool):.2f}")
    axes[0].axvline(np.median(OSI_pool), color=color, lw=2, ls="--")
    axes[1].hist(DSI_pool, bins=bins, color=color, alpha=0.6,
                 label=f"{area_name}  median={np.median(DSI_pool):.2f}")
    axes[1].axvline(np.median(DSI_pool), color=color, lw=2, ls="--")

axes[0].set(xlabel="OSI", ylabel="ROI count", title="Orientation Selectivity by Area")
axes[0].legend(fontsize=9); axes[0].spines[["top","right"]].set_visible(False)
axes[1].set(xlabel="DSI", ylabel="ROI count", title="Direction Selectivity by Area")
axes[1].legend(fontsize=9); axes[1].spines[["top","right"]].set_visible(False)

fig.suptitle("Mesoscope VISp vs VISl — OSI & DSI  (sub-839909, all planes pooled)",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

ori_handle.close()

---
## Section 4 — Cross-session Population Summary

All 3 techniques processed across every available session (42 mesoscope, 6 ecephys, 9 SLAP2).  
Pre-computed statistics are loaded directly from the repository — no long-running collection needed.

| Technique | Sessions | Dandiset |
|---|---|---|
| Mesoscope | 42 sessions, 6 subjects | 001768 |
| Ecephys (Neuropixels) | 6 sessions, 2 subjects | 001637 |
| SLAP2 | 9 sessions, 4 subjects | 001424 |

**OSI** = Orientation Selectivity Index (vector-sum, 0 = untuned, 1 = perfectly tuned)  
**DSI** = Direction Selectivity Index (same method, 1θ instead of 2θ)

In [ ]:
import pandas as pd

# ── Load precomputed CSVs from GitHub ─────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/maierav/claupenscope/main/results/"

meso_df  = pd.read_csv(BASE_URL + "meso_ori_stats.csv")
ephys_df = pd.read_csv(BASE_URL + "ephys_ori_stats.csv")
slap2_df = pd.read_csv(BASE_URL + "slap2_ori_stats.csv")

# Mesoscope: session-level aggregate (mean of per-plane means, per area)
sess_area = (
    meso_df.groupby(["asset_id", "subject_id", "area"])
    .agg(mean_OSI=("mean_OSI", "mean"), mean_DSI=("mean_DSI", "mean"))
    .reset_index()
)

print("Mesoscope:", meso_df["asset_id"].nunique(), "sessions,",
      meso_df["subject_id"].nunique(), "subjects")
print("Ecephys  :", ephys_df["asset_id"].nunique(), "sessions,",
      ephys_df["subject_id"].nunique(), "subjects,",
      ephys_df["probe"].nunique(), "probes")
print("SLAP2    :", slap2_df["asset_id"].nunique(), "sessions,",
      slap2_df["subject_id"].nunique(), "subjects")

# ── Figure 4: 3-panel comparison across techniques ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ─ Panel A: Mesoscope VISp vs VISl ────────────────────────────────────
ax = axes[0]
MESO_COLORS = {"VISp": "#4878CF", "VISl": "#D65F5F"}
for xi, area in enumerate(["VISp", "VISl"], start=1):
    vals = sess_area.loc[sess_area["area"] == area, "mean_OSI"].values
    parts = ax.violinplot([vals], positions=[xi], showmedians=False, showextrema=False)
    parts["bodies"][0].set_facecolor(MESO_COLORS[area]); parts["bodies"][0].set_alpha(0.5)
    jx = xi + np.random.default_rng(0).uniform(-0.08, 0.08, len(vals))
    ax.scatter(jx, vals, color=MESO_COLORS[area], s=25, alpha=0.8, zorder=3, edgecolors="none")
    ax.plot([xi - 0.2, xi + 0.2], [np.mean(vals)] * 2, color="k", lw=2.5, zorder=4)
ax.set_xticks([1, 2]); ax.set_xticklabels(["VISp", "VISl"])
ax.set_ylabel("Mean OSI (session average)")
ax.set_title("Mesoscope\n42 sessions · 6 subjects", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
for area, color in MESO_COLORS.items():
    ax.annotate(f"{area}\n{sess_area[sess_area['area']==area]['mean_OSI'].mean():.3f}",
                xy=({"VISp": 1, "VISl": 2}[area], 0.43), ha="center", fontsize=8,
                color=color, fontweight="bold")

# ─ Panel B: Ecephys per probe ─────────────────────────────────────────
ax = axes[1]
probes = sorted(ephys_df["probe"].unique())
PROBE_COLOR = "#7B7B7B"
for pi, probe in enumerate(probes):
    vals = ephys_df.loc[ephys_df["probe"] == probe, "mean_OSI"].values
    jx = pi + np.random.default_rng(0).uniform(-0.18, 0.18, len(vals))
    ax.scatter(jx, vals, color=PROBE_COLOR, s=30, alpha=0.75, zorder=3, edgecolors="none")
    ax.plot([pi - 0.28, pi + 0.28], [np.mean(vals)] * 2, color="k", lw=2.5, zorder=4)
ax.set_xticks(range(len(probes))); ax.set_xticklabels(probes, rotation=20, ha="right")
ax.set_ylabel("Mean OSI (session average)")
ax.set_title("Ecephys (Neuropixels)\n6 sessions · 2 subjects · 6 probes", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)

# ─ Panel C: SLAP2 DMD1 vs DMD2 ───────────────────────────────────────
ax = axes[2]
DMD_COLORS = {"DMD1": "#4878CF", "DMD2": "#D65F5F"}
dmds = sorted(slap2_df["dmd"].unique())
for xi, dmd in enumerate(dmds, start=1):
    vals = slap2_df.loc[slap2_df["dmd"] == dmd, "mean_OSI"].values
    parts = ax.violinplot([vals], positions=[xi], showmedians=False, showextrema=False)
    parts["bodies"][0].set_facecolor(DMD_COLORS.get(dmd, "gray"))
    parts["bodies"][0].set_alpha(0.5)
    jx = xi + np.random.default_rng(0).uniform(-0.08, 0.08, len(vals))
    ax.scatter(jx, vals, color=DMD_COLORS.get(dmd, "gray"),
               s=30, alpha=0.85, zorder=3, edgecolors="none")
    ax.plot([xi - 0.2, xi + 0.2], [np.mean(vals)] * 2, color="k", lw=2.5, zorder=4)
ax.set_xticks(range(1, len(dmds) + 1)); ax.set_xticklabels(dmds)
ax.set_ylabel("Mean OSI (session average)")
ax.set_title("SLAP2\n9 sessions · 4 subjects", fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
for xi, dmd in enumerate(dmds, start=1):
    vals = slap2_df.loc[slap2_df["dmd"] == dmd, "mean_OSI"].values
    ax.annotate(f"{dmd}\n{np.mean(vals):.3f}", xy=(xi, ax.get_ylim()[0]),
                ha="center", va="bottom", fontsize=8,
                color=DMD_COLORS.get(dmd, "gray"), fontweight="bold")

fig.suptitle(
    "Cross-session orientation selectivity — all 3 techniques\n"
    "Black bar = cross-session mean · dots = individual sessions",
    fontsize=11, fontweight="bold",
)
plt.tight_layout()
plt.show()

# ── Quick summary table ───────────────────────────────────────────────
print("\n── Summary: Mean OSI across sessions ──")
print(f"  Mesoscope  VISp : {sess_area[sess_area['area']=='VISp']['mean_OSI'].mean():.3f} "
      f"± {sess_area[sess_area['area']=='VISp']['mean_OSI'].sem():.3f} SEM")
print(f"  Mesoscope  VISl : {sess_area[sess_area['area']=='VISl']['mean_OSI'].mean():.3f} "
      f"± {sess_area[sess_area['area']=='VISl']['mean_OSI'].sem():.3f} SEM")
print(f"  Ecephys  (all)  : {ephys_df['mean_OSI'].mean():.3f} "
      f"± {ephys_df['mean_OSI'].sem():.3f} SEM  ({ephys_df['n_units'].sum():,} units total)")
print(f"  SLAP2    DMD1   : {slap2_df[slap2_df['dmd']=='DMD1']['mean_OSI'].mean():.3f} "
      f"± {slap2_df[slap2_df['dmd']=='DMD1']['mean_OSI'].sem():.3f} SEM")
print(f"  SLAP2    DMD2   : {slap2_df[slap2_df['dmd']=='DMD2']['mean_OSI'].mean():.3f} "
      f"± {slap2_df[slap2_df['dmd']=='DMD2']['mean_OSI'].sem():.3f} SEM")